# Gurobi em 800 setores — execução em nuvem (Kaggle/Colab, licença WLS acadêmica)

Roda **só o MIP** (`compare_solvers mip`) na instância de 800 setores (e na de 400, complemento
de escala) com os parâmetros de memória do Gurobi, e empacota a pasta de saída para devolver.

**Antes de rodar:**
1. Gere a licença **WLS Academic** no portal da Gurobi (na rede da UNIFESP ou via VPN). Ela é
   diferente da Named-User e vale 90 dias.
2. Cadastre três segredos no notebook: `WLSACCESSID`, `WLSSECRET`, `LICENSEID`
   (Kaggle: *Add-ons → Secrets*; Colab: ícone da chave 🔑 na barra lateral).
3. Kaggle: notebook **CPU** (≈30 GB de RAM), *Internet on*. Colab grátis tem ≈12 GB.
4. Disponibilize o código: suba um zip do repositório como *Dataset* (Kaggle) / arquivo (Colab)
   e ajuste `REPO_ZIP`, ou preencha `GIT_URL` + segredo `GITHUB_TOKEN` (repositório privado).

Saída: `scale800_cloud_outputs.zip` (JSON por solve, `gurobi/*.log`, soluções intermediárias
de `SolFiles` e `env.json` com RAM/núcleos/versões).

In [ ]:
# --- ajuste aqui ---
REPO_ZIP = ""  # ex.: "/kaggle/input/cap-repo/capacitated-assignment-problem.zip"
GIT_URL = ""  # ex.: "github.com/unifesp-ita-natura/capacitated-assignment-problem.git"
GIT_BRANCH = "fine-tuning"
CONFIG = "configs/experiments/sa_calibration/scale800_cloud.yaml"
WORKDIR = "/kaggle/working" if __import__("os").path.isdir("/kaggle/working") else "/content"

In [ ]:
import os


def secret(name: str) -> str:
    """Lê um segredo do Kaggle ou do Colab (nunca imprima o valor)."""
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(name)
    except ImportError:
        from google.colab import userdata

        return userdata.get(name)


license_path = os.path.join(WORKDIR, "gurobi.lic")
with open(license_path, "w") as f:
    for key in ("WLSACCESSID", "WLSSECRET", "LICENSEID"):
        f.write(f"{key}={secret(key)}\n")
os.chmod(license_path, 0o600)
os.environ["GRB_LICENSE_FILE"] = license_path
print("licença WLS gravada em", license_path)

In [ ]:
import subprocess
import zipfile

repo = os.path.join(WORKDIR, "capacitated-assignment-problem")
if REPO_ZIP:
    with zipfile.ZipFile(REPO_ZIP) as z:
        z.extractall(WORKDIR)
elif GIT_URL:
    token = secret("GITHUB_TOKEN")
    url = f"https://{token}@{GIT_URL}"
    subprocess.run(["git", "clone", "-b", GIT_BRANCH, url, repo], check=True)
else:
    raise SystemExit("defina REPO_ZIP ou GIT_URL")
os.chdir(repo)
print(os.getcwd())

In [ ]:
!pip install -q uv
!uv python install 3.13
!uv sync --python 3.13
!free -h && nproc

In [ ]:
# confere a licença antes da hora de execução
!uv run python -c "import gurobipy as gp; gp.Model(); print(gp.gurobi.version(), 'licença OK')"

In [ ]:
# o MIP de cada instância, um por vez (até 1 h cada); a saída vai sendo gravada por solve
!uv run python -m experiments.sa_calibration.compare_solvers mip --config {CONFIG} \
    2>&1 | tee mip_cloud.log

In [ ]:
import shutil

import yaml

out = yaml.safe_load(open(CONFIG))["output_dir"]
shutil.copy("mip_cloud.log", out)
archive = shutil.make_archive(os.path.join(WORKDIR, "scale800_cloud_outputs"), "zip", out)
print("baixe:", archive)